In [1]:
import numpy as np
import pandas as pd
from sklearn.feature_selection import mutual_info_classif

#Calculate mutual information (manually from a table)

#Counts for a binary feature X and binary target Y
counts = pd.DataFrame(
    [[40, 10],
     [10, 40]],
    index=["X=0", "X=1"],
    columns=["Y=0", "Y=1"]
)

print("Counts:")
display(counts)

#Convert counts to joint probabilities
p_xy = counts / counts.values.sum()

#Marginal probabilities
p_x = p_xy.sum(axis=1)
p_y = p_xy.sum(axis=0)

print("Joint probabilities:")
display(p_xy)

print("Marginal probabilities for X:")
display(p_x)

print("Marginal probabilities for Y:")
display(p_y)

#Mutual information calculation
mi = 0

for x in p_xy.index:
    for y in p_xy.columns:
        joint = p_xy.loc[x, y]
        expected_if_independent = p_x.loc[x] * p_y.loc[y]
        
        mi += joint * np.log2(joint / expected_if_independent)

print(f"Manual mutual information: {mi:.4f} bits")


#Use mutual information for feature selection

np.random.seed(1)

n = 200

X = pd.DataFrame({
    "useful_feature": np.random.binomial(1, 0.5, n),
    "mostly_noise": np.random.normal(0, 1, n)
})

#Target is mostly determined by useful_feature
y = X["useful_feature"].copy()

#Flip 15% of target values to add noise
flip = np.random.binomial(1, 0.15, n)
y = np.where(flip == 1, 1 - y, y)

#Calculate mutual information scores
mi_scores = mutual_info_classif(X, y, random_state=1)

results = pd.DataFrame({
    "feature": X.columns,
    "mutual_information": mi_scores
}).sort_values("mutual_information", ascending=False)

print("Feature selection results:")
display(results)

Counts:


,Y=0,Y=1
X=0,40,10
X=1,10,40


Joint probabilities:


,Y=0,Y=1
X=0,0.4,0.1
X=1,0.1,0.4


Marginal probabilities for X:


X=0    0.5
X=1    0.5
dtype: float64

Marginal probabilities for Y:


Y=0    0.5
Y=1    0.5
dtype: float64

Manual mutual information: 0.2781 bits
Feature selection results:


,feature,mutual_information
0,useful_feature,0.251614
1,mostly_noise,0.059304
